# 00: Benchmark summary

*How do the knowledge graphs compare across the whole benchmark, and do the seven dimensions order them the same way?*

| Display item | What it shows | Built from |
|---|---|---|
| **Quality profiles** | Every dimension score as a polar profile per KG | `summary_scores` |
| **Category scores** | Mean score per category (Content / Structure / Inference) | `category_scores` |
| **Metric heatmap** | Every sub-metric beside the dimension scores and their mean | `SUB_METRICS` |
| **Summary table** | The same matrix as CSV, at full precision | `summary_df` |
| **Concordance** | Kendall's *W* under five specifications, rank dispersion, dissociation counts | `summary_df` |

Nothing is measured here. Every value is read from the checkpoint its dimension notebook wrote, so this notebook inherits their panel exactly: a dimension that has not been run, or a graph a dimension could not score, is carried as N/A and never as zero.

**Reported, not scored.** The permutation *p*-value for Kendall's *W*. The sub-metrics are not independent judges, so the null is optimistic; the effect size and its stability across specifications are the quantities the manuscript reports.

**Inputs:** `config/kgs.yaml` · `results/checkpoints/01_coverage.pkl` … `results/checkpoints/07_generalization.pkl`

**Outputs:** `results/figures/00_summary/` · `results/tables/00_summary/`

Design notes and justifications: [`README.md`](README.md)

## Set-up

Checkpoints are read from `results/checkpoints/`, one per dimension notebook. A missing checkpoint stops the run rather than dropping a spoke from the figures, unless a partial summary is asked for in the config cell. KG names come from `config/kgs.yaml`, so adding or removing a graph there needs no change in this notebook.

In [ ]:
# Pick up edits to src/ without restarting the kernel. Without this, editing a
# module after it has been imported leaves the kernel holding the old version,
# which surfaces as a confusing ImportError for a name that is plainly on disk.
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except Exception:
    pass                      # not running under IPython (headless runner, plain python)

import math
import sys
import warnings
from itertools import combinations, product
from pathlib import Path

warnings.filterwarnings('ignore')

# Locate the repo root from any depth, then make the package importable.
# (`pip install -e .` also works and makes this block a no-op.)
_root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / 'config' / 'kgs.yaml').exists())
if str(_root / 'src') not in sys.path:
    sys.path.insert(0, str(_root / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.patches import Rectangle

from biokgsuite.io import Outputs, assert_full_panel, find_config, load_config
from biokgsuite.viz.style import (ALERT_RED, DOUBLE_COL_W, HEATMAP_CMAP, KG_PALETTE,
                                  TEXT_COLOR, TICK_COLOR, clean_ax, setup_style)

setup_style()

In [ ]:
# This notebook reads no analysis parameter of its own: every number comes from
# a dimension checkpoint, and the only choices made here are which graphs to show
# and whether an incomplete benchmark may be summarised at all.
config = load_config(find_config(_root))
BASE   = config['_base_dir']

# Every figure and table goes through `out`, so results land in
# results/{figures,tables}/00_summary/, and `out.load_checkpoint` reads the
# dimension pickles from results/checkpoints/.
out = Outputs('00_summary', BASE)

KG_NAMES  = list(config['knowledge_graphs'])
KG_COLORS = {name: KG_PALETTE.get(name, '#888888') for name in KG_NAMES}

# A summary over fewer dimensions than 01-07, or over fewer graphs than
# config/kgs.yaml declares, is a different experiment rather than a smaller one,
# so it has to be asked for. The panel that was actually summarised is printed by
# the loading cell either way.
ALLOW_PARTIAL_DIMENSIONS = False
ALLOW_PARTIAL_PANEL      = False

### Loading

In [ ]:
# One checkpoint per dimension notebook. `out.load_checkpoint` names the notebook
# to run when a file is absent, so a miss is caught here rather than surfacing as
# an N/A column three figures later.
CKPT_NOTEBOOKS = {
    'coverage':            '01_coverage',
    'annotation_accuracy': '02_annotation_accuracy',
    'trustworthiness':     '03_trustworthiness',
    'topology':            '04_topology',
    'stability':           '05_stability',
    'task_performance':    '06_task_performance',
    'generalisation':      '07_generalization',
}

ckpts = {}
for _dim, _nb in CKPT_NOTEBOOKS.items():
    try:
        ckpts[_dim] = out.load_checkpoint(_nb)
    except FileNotFoundError:
        print(f'{_dim:22s} [NOT FOUND] {_nb}.pkl')

DIMS         = list(CKPT_NOTEBOOKS)
LOADED_DIMS  = list(ckpts)
_missing_dim = [d for d in DIMS if d not in ckpts]
print(f'{len(LOADED_DIMS)}/{len(DIMS)} dimensions loaded: {", ".join(LOADED_DIMS)}')

# Refuse to summarise an incomplete benchmark.
#
# An absent checkpoint drops a spoke from the radar, a block from the heatmap and
# a judge from the concordance, and the overall mean is then taken over a
# different set of dimensions than the manuscript describes. All of it still
# renders, and none of it says so on the figure.
if _missing_dim and not ALLOW_PARTIAL_DIMENSIONS:
    raise RuntimeError(
        f'{len(_missing_dim)} of {len(DIMS)} dimension checkpoints missing: '
        f'{", ".join(_missing_dim)}. Run those notebooks, or set '
        'ALLOW_PARTIAL_DIMENSIONS = True in the config cell to summarise a subset.')

# The same argument one level down. Each checkpoint carries the graphs its own
# notebook managed to score, so a dimension that saw two of six contributes two
# rows here while the table around it looks complete.
SCORED_KGS = [kg for kg in KG_NAMES
              if any(kg in _ck.get('summary_scores', {}) for _ck in ckpts.values())]
PANEL = assert_full_panel(SCORED_KGS, config,
                          allow_partial=ALLOW_PARTIAL_PANEL,
                          notebook='00_summary')

for _dim, _ck in ckpts.items():
    _got = [kg for kg in KG_NAMES if kg in _ck.get('summary_scores', {})]
    if len(_got) < len(KG_NAMES):
        print(f'{_dim:22s} scored {len(_got)}/{len(KG_NAMES)} graphs: {", ".join(_got)}')

## 1. Score matrix

In [ ]:
CATEGORIES = {
    'Content':   ['coverage', 'annotation_accuracy', 'trustworthiness'],
    'Structure': ['topology', 'stability'],
    'Inference': ['task_performance', 'generalisation'],
}

DIM_LABELS = {
    'coverage':            'Coverage',
    'annotation_accuracy': 'Annotation accuracy',
    'trustworthiness':     'Trustworthiness',
    'topology':            'Topology',
    'stability':           'Stability',
    'task_performance':    'Task performance',
    'generalisation':      'Generalisation',
}

# summary_scores[kg][dim]: the dimension score its notebook wrote, on 0-1.
# A dimension a graph was not scored on is absent rather than 0.0, so it stays
# out of every mean taken below instead of being averaged in as a failure.
summary_scores = {kg: {} for kg in KG_NAMES}
for _dim, _ck in ckpts.items():
    for _kg in KG_NAMES:
        _v = _ck['summary_scores'].get(_kg)
        if _v is not None:
            summary_scores[_kg][_dim] = float(_v)

# category_scores[kg][cat]: mean over the loaded dimensions of that category.
category_scores = {}
for _kg in KG_NAMES:
    category_scores[_kg] = {}
    for _cat, _cat_dims in CATEGORIES.items():
        _vals = [summary_scores[_kg][_d] for _d in _cat_dims if _d in summary_scores[_kg]]
        category_scores[_kg][_cat] = round(float(np.mean(_vals)), 4) if _vals else None

dim_score_df = pd.DataFrame(
    [{'KG': _kg, **{DIM_LABELS[_d]: summary_scores[_kg].get(_d) for _d in DIMS}}
     for _kg in KG_NAMES]).set_index('KG')
out.table(dim_score_df.reset_index(), 'dimension_scores')
print('Dimension scores (0-1, higher is better). N/A = not scored, not zero.')
display(dim_score_df)

category_df = pd.DataFrame(
    [{'KG': _kg, **{_c: category_scores[_kg][_c] for _c in CATEGORIES}}
     for _kg in KG_NAMES]).set_index('KG')
out.table(category_df.reset_index(), 'category_scores')
print('Category scores (mean over the loaded dimensions of each category).')
display(category_df)

## 2. Quality profiles

One polygon per graph over the seven dimension scores. A dimension a graph was not scored on breaks the polygon rather than closing it at the origin, so an unmeasured dimension cannot be read as a zero.

In [ ]:
# Two-line labels for the crowded spokes; the rest are the sub-metric names
# the dimension notebooks use.
_SPOKE_LABELS = {
    'coverage':            'Coverage',
    'annotation_accuracy': 'Annotation\naccuracy',
    'trustworthiness':     'Trustworthiness',
    'topology':            'Topology',
    'stability':           'Stability',
    'task_performance':    'Task\nperformance',
    'generalisation':      'Generalisation',
}

_avail_dims = [d for d in DIMS if d in LOADED_DIMS]
_labels     = [_SPOKE_LABELS.get(d, DIM_LABELS.get(d, d)) for d in _avail_dims]
N           = len(_avail_dims)
angles      = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles_c    = angles + angles[:1]

# No graph is emphasised on the headline figure. Drawing one series
# heavier than the rest puts a thumb on the comparison a benchmark exists
# to make, and it is the authors' own graph that was being emphasised.
HIGHLIGHT = None

fig, ax = plt.subplots(figsize=(7.2, 7.2), subplot_kw={'polar': True})
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)

# Radial axis.
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['0.25', '0.50', '0.75', '1.00'], fontsize=8, color=TICK_COLOR)
# tuck the radial numbers into the gap just before the first spoke
ax.set_rlabel_position(np.degrees(angles[0] - np.pi / N))
for lab in ax.get_yticklabels():               # white halo so they never clash
    lab.set_bbox(dict(boxstyle='round,pad=0.12', fc='white', ec='none', alpha=0.75))
ax.spines['polar'].set_visible(False)
ax.grid(color='#E6E6E6', lw=0.6)

# One polygon per graph.
for _kg in KG_NAMES:
    # np.nan, not 0.0: a dimension whose checkpoint is absent was not measured,
    # and drawing it at the origin publishes 'not measured' as 'worst possible'
    # on the headline figure. matplotlib breaks the polygon at a NaN vertex.
    _vals   = [summary_scores[_kg].get(d, np.nan) for d in _avail_dims]
    _closed = _vals + _vals[:1]
    is_hi   = (_kg == HIGHLIGHT)
    ax.plot(angles_c, _closed, color=KG_COLORS[_kg],
            lw=2.6 if is_hi else 1.5,
            alpha=1.0 if is_hi else 0.85,
            zorder=5 if is_hi else 2, label=_kg.upper())
    ax.fill(angles_c, _closed, color=KG_COLORS[_kg],
            alpha=0.12 if is_hi else 0.04, zorder=1)

# Perimeter labels, each aligned to grow outward from the plot.
ax.set_xticks(angles)
ax.set_xticklabels(_labels, fontsize=11, color=TEXT_COLOR)
ax.tick_params(axis='x', pad=4)
for lab, ang in zip(ax.get_xticklabels(), angles):
    sx, sy = np.cos(np.pi/2 - ang), np.sin(np.pi/2 - ang)
    lab.set_horizontalalignment('left' if sx > 0.10 else 'right' if sx < -0.10 else 'center')
    lab.set_verticalalignment('bottom' if sy > 0.10 else 'top' if sy < -0.10 else 'center')

# Legend and title.
ax.legend(fontsize=9, frameon=False, bbox_to_anchor=(0.5, -0.04),
          loc='upper center', ncol=3, columnspacing=1.6, handletextpad=0.6)
ax.set_title('Multi-dimensional quality profiles of biomedical KGs',
             fontsize=13, color=TEXT_COLOR, pad=10)

fig.subplots_adjust(top=0.90, bottom=0.12, left=0.08, right=0.92)
out.figure(fig, 'quality_profiles')
plt.show()

## 3. Category scores

In [ ]:
# Grouped bars: one group per category, one bar per graph.
# The reference lines at 0.5 and 0.8 are reading aids, not thresholds any
# score is judged against; nothing downstream uses them.
def _style_left_spine(ax):
    # Only the left spine carries information once the grid is drawn.
    for sp in ('top', 'right', 'bottom'):
        ax.spines[sp].set_visible(False)
    ax.spines['left'].set_color('#333333')
    ax.spines['left'].set_linewidth(0.5)

_cats = list(CATEGORIES.keys())
_x    = np.arange(len(_cats))
_w    = 0.8 / max(len(KG_NAMES), 1)

fig, ax = plt.subplots(figsize=(DOUBLE_COL_W * 0.65, 3.1))

for _i, _kg in enumerate(KG_NAMES):
    _vals = [category_scores[_kg].get(_c) for _c in _cats]
    _xs   = _x + (_i - (len(KG_NAMES) - 1) / 2) * _w
    _bars = ax.bar(_xs, [np.nan if _v is None else _v for _v in _vals], _w * 0.9,
                   color=KG_COLORS[_kg], label=_kg.upper(),
                   edgecolor='white', linewidth=0.5)
    for _bar, _v in zip(_bars, _vals):
        if _v is not None:
            ax.text(_bar.get_x() + _bar.get_width() / 2, _bar.get_height() + 0.015,
                    f'{_v:.2f}', ha='center', va='bottom', rotation=90,
                    fontsize=7, color=TEXT_COLOR)

ax.axhline(0.5, color='#868074', ls='--', lw=0.9, alpha=0.7, zorder=0)
ax.axhline(0.8, color='#655F55', ls='--', lw=0.9, alpha=0.7, zorder=0)
ax.set_xticks(_x)
ax.set_xticklabels(_cats, fontsize=9.5, color=TICK_COLOR)
ax.set_ylim(0, 1.18)
ax.set_yticks(np.arange(0, 1.01, 0.2))
ax.set_ylabel('Mean score', fontsize=9, color=TICK_COLOR)
ax.margins(x=0.02)

ax.legend(fontsize=7.5, frameon=False, ncol=3, loc='upper center',
          bbox_to_anchor=(0.5, -0.13), columnspacing=1.4,
          handlelength=1.2, handletextpad=0.5)

_style_left_spine(ax)
clean_ax(ax, title='Aggregate category scores across evaluation dimensions')
plt.tight_layout()
out.figure(fig, 'category_scores')
plt.show()

## 4. Metric and dimension heatmap

Every sub-metric, with the dimension scores and their mean set off to the right. N/A cells are shaded, not coloured on the score scale.

In [ ]:
# Every sub-metric the seven dimension notebooks score, in their order, under
# the names their own title cells give them. Each is read straight out of the
# dimension's `sub_scores`, so a key renamed in a dimension notebook shows up
# here as a blank column rather than as a silently different number.
# 17 sub-metric definitions (column_label, dim_key, extractor_fn(ck, kg))
SUB_METRICS = [
    # Coverage (2)
    ('Entity coverage',             'coverage',            lambda ck, kg: ck.get('sub_scores', {}).get(kg, {}).get('entity_coverage')),
    ('Relation coverage',           'coverage',            lambda ck, kg: ck.get('sub_scores', {}).get(kg, {}).get('relation_coverage')),
    # Annotation accuracy (2)
    ('Entity validity',             'annotation_accuracy', lambda ck, kg: ck.get('sub_scores', {}).get(kg, {}).get('entity_validity')),
    ('Relational consistency',      'annotation_accuracy', lambda ck, kg: ck.get('sub_scores', {}).get(kg, {}).get('relational_consistency')),
    # Trustworthiness (3)
    ('Edge traceability',           'trustworthiness',     lambda ck, kg: ck.get('sub_scores', {}).get(kg, {}).get('edge_traceability')),
    ('Source diversity',            'trustworthiness',     lambda ck, kg: ck.get('sub_scores', {}).get(kg, {}).get('source_diversity')),
    ('Uncertainty quantification',  'trustworthiness',     lambda ck, kg: ck.get('sub_scores', {}).get(kg, {}).get('uq_coverage')),
    # Topology (4)
    ('Graph cohesion',              'topology',            lambda ck, kg: ck.get('sub_scores', {}).get(kg, {}).get('connectedness')),
    ('Clustering coefficient',      'topology',            lambda ck, kg: ck.get('sub_scores', {}).get(kg, {}).get('small_world')),
    ('Graph reachability',          'topology',            lambda ck, kg: ck.get('sub_scores', {}).get(kg, {}).get('reachability')),
    ('Community purity',            'topology',            lambda ck, kg: ck.get('sub_scores', {}).get(kg, {}).get('community_purity')),
    # Stability (2)
    ('Random-dropout stability',    'stability',           lambda ck, kg: ck.get('sub_scores', {}).get(kg, {}).get('random_stability')),
    ('Periphery-dropout stability', 'stability',           lambda ck, kg: ck.get('sub_scores', {}).get(kg, {}).get('periphery_stability')),
    # Task performance (3)
    ('Link prediction',             'task_performance',    lambda ck, kg: ck.get('sub_scores', {}).get(kg, {}).get('link_prediction')),
    ('Neighbourhood retrieval',     'task_performance',    lambda ck, kg: ck.get('sub_scores', {}).get(kg, {}).get('nbhd_retrieval')),
    ('Multi-hop reasoning',         'task_performance',    lambda ck, kg: ck.get('sub_scores', {}).get(kg, {}).get('multihop_reasoning')),
    # Generalisation (3)
    ('Data-sparse generalisation',  'generalisation',      lambda ck, kg: ck.get('sub_scores', {}).get(kg, {}).get('data_sparse_gen')),
    ('Cross-domain generalisation', 'generalisation',      lambda ck, kg: ck.get('sub_scores', {}).get(kg, {}).get('cross_domain_gen')),
    ('Prospective generalisation',  'generalisation',      lambda ck, kg: ck.get('sub_scores', {}).get(kg, {}).get('temporal_gen')),
]
print(f'{len(SUB_METRICS)} sub-metrics across {len({_d for _, _d, _ in SUB_METRICS})} dimensions.')

# Build metric matrix (rows=KGs, cols=sub-metrics)
heat_data = np.full((len(KG_NAMES), len(SUB_METRICS)), np.nan)
for _j, (_, _dim, _extractor) in enumerate(SUB_METRICS):
    _ck = ckpts.get(_dim)
    if _ck is None:
        continue
    for _i, _kg in enumerate(KG_NAMES):
        _v = _extractor(_ck, _kg)
        if _v is not None and not (isinstance(_v, float) and math.isnan(_v)):
            heat_data[_i, _j] = float(_v)

# Dimension-level scores, recomputed here as the unweighted mean of the
# sub-metrics above, so the block beside the heatmap is arithmetic the reader
# can check against the cells to its left.
DIM_ORDER = []
for _, _dim, _ in SUB_METRICS:
    if _dim not in DIM_ORDER:
        DIM_ORDER.append(_dim)

_dim_cols = {d: [j for j, (_, dd, _) in enumerate(SUB_METRICS) if dd == d]
             for d in DIM_ORDER}

dim_data = np.full((len(KG_NAMES), len(DIM_ORDER) + 1), np.nan)
for _i in range(len(KG_NAMES)):
    for _d_idx, _d in enumerate(DIM_ORDER):
        _vals = heat_data[_i, _dim_cols[_d]]
        _vals = _vals[~np.isnan(_vals)]
        if _vals.size:
            dim_data[_i, _d_idx] = _vals.mean()
    _dims = dim_data[_i, :len(DIM_ORDER)]
    _dims = _dims[~np.isnan(_dims)]
    if _dims.size:
        dim_data[_i, -1] = _dims.mean()          # Overall mean

DIM_COL_LABELS = [DIM_LABELS.get(_d, _d) for _d in DIM_ORDER] + ['Overall']

# Layout: the metric block, a gap, then the dimension block.
_N_metric = len(SUB_METRICS)
_N_dim    = dim_data.shape[1]
_GAP      = 1.2
_dim_off  = _N_metric + _GAP
_N_total  = _N_metric + _GAP + _N_dim
_N_rows   = len(KG_NAMES)

_fig_w = max(DOUBLE_COL_W, _N_total * 0.54 + 1.8)
_fig_h = _N_rows * 0.58 + 3.2
fig, ax = plt.subplots(figsize=(_fig_w, _fig_h))

_norm     = Normalize(vmin=0, vmax=1)
_na_color = '#EDEADF'

def _draw_block(mat, x_offset):
    for _i in range(mat.shape[0]):
        for _j in range(mat.shape[1]):
            _v  = mat[_i, _j]
            _x  = _j + x_offset
            _fc = _na_color if np.isnan(_v) else HEATMAP_CMAP(_norm(_v))
            ax.add_patch(Rectangle((_x - 0.5, _i - 0.5), 1, 1,
                                   facecolor=_fc, edgecolor=_fc, linewidth=0.5))
            if not np.isnan(_v):
                _col = 'white' if _v >= 0.65 else TEXT_COLOR
                ax.text(_x, _i, f'{_v:.2f}', ha='center', va='center',
                        fontsize=7.5, fontweight='bold', color=_col)
            else:
                ax.text(_x, _i, 'N/A', ha='center', va='center',
                        fontsize=6.5, color='#A8A296')

_draw_block(heat_data, 0)
_draw_block(dim_data, _dim_off)

ax.set_xlim(-0.5, _N_total - 0.5)
ax.set_ylim(_N_rows - 0.5, -0.5)
ax.set_aspect('equal')

# Dimension group labels above the metric block
_dim_groups, _prev, _gstart = [], None, 0
for _j, (_, _d, _) in enumerate(SUB_METRICS):
    if _d != _prev and _prev is not None:
        _dim_groups.append((_prev, _gstart, _j - 1))
        _gstart = _j
    _prev = _d
_dim_groups.append((_prev, _gstart, _N_metric - 1))

for _d, _js, _je in _dim_groups:
    ax.text((_js + _je) / 2.0, -0.75, DIM_LABELS.get(_d, _d),
            ha='center', va='bottom', fontsize=7.5,
            fontweight='bold', color=TEXT_COLOR, clip_on=False)

# Header over the dimension block
ax.text(_dim_off + (_N_dim - 1) / 2.0, -0.75, 'Dimension scores',
        ha='center', va='bottom', fontsize=7.5,
        fontweight='bold', color=TEXT_COLOR, clip_on=False)

# Ticks across both blocks
_xt = list(range(_N_metric)) + [_dim_off + _j for _j in range(_N_dim)]
_xl = [_m[0] for _m in SUB_METRICS] + DIM_COL_LABELS
ax.set_xticks(_xt)
ax.set_xticklabels(_xl, rotation=45, ha='right', fontsize=8, color=TICK_COLOR)
ax.set_yticks(range(_N_rows))
ax.set_yticklabels([_kg.upper() for _kg in KG_NAMES], fontsize=9, color=TICK_COLOR)
ax.tick_params(length=0)
for _sp in ax.spines.values():
    _sp.set_visible(False)

_sm = ScalarMappable(cmap=HEATMAP_CMAP, norm=_norm)
_sm.set_array([])
plt.colorbar(_sm, ax=ax, shrink=0.55, pad=0.01, label='Score (0-1)')

ax.set_title('Metric and dimension scores across knowledge graphs',
             fontsize=11, fontweight='bold', color=TEXT_COLOR, pad=42)
plt.tight_layout()
out.figure(fig, 'metric_heatmap')
plt.show()

## 5. Summary table

In [ ]:
# The table the manuscript quotes: KGs x (every sub-metric, every dimension
# score, and their mean). Written at full precision; rounding belongs in the
# figure and the manuscript, not in the file a reader recomputes from.
_DIM_SCORE_SUFFIX = ' [dim]'
_OVERALL_COL      = 'Overall Mean'

# Columns: sub-metrics, then dimension scores, then the overall mean.
_sub_cols = [_m[0] for _m in SUB_METRICS]
_dim_cols = [DIM_LABELS.get(_d, _d) + _DIM_SCORE_SUFFIX for _d in DIMS]
_all_cols = _sub_cols + _dim_cols + [_OVERALL_COL]

_score_rows = []
for _kg in KG_NAMES:
    _row = {}
    # Sub-metrics
    for _col_label, _dim, _extractor in SUB_METRICS:
        _ck = ckpts.get(_dim)
        _v  = _extractor(_ck, _kg) if _ck is not None else None
        _row[_col_label] = _v
    # Dim scores
    for _d in DIMS:
        _row[DIM_LABELS.get(_d, _d) + _DIM_SCORE_SUFFIX] = summary_scores[_kg].get(_d)
    # Overall mean across loaded dimensions
    _dim_vals = [summary_scores[_kg].get(_d) for _d in DIMS if _d in summary_scores[_kg]]
    _row[_OVERALL_COL] = round(float(np.mean(_dim_vals)), 4) if _dim_vals else None
    _score_rows.append(_row)

# float, not object: an unscored cell arrives as None, and None propagates
# through a subtraction as a TypeError instead of as the NaN every guard
# downstream tests for.
summary_df = pd.DataFrame(_score_rows, index=KG_NAMES, columns=_all_cols).astype(float)
summary_df.index.name = 'KG'

out.table(summary_df.reset_index(), 'benchmark_summary')

# Displayed with values below 0.5 in red. The 0.5 is a reading aid for the
# screen only; the exported CSV carries no such mark and nothing is scored
# against it.
def _fmt_cell(v):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return 'N/A'
    return f'{v:.3f}'

def _highlight_low(v):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return ''
    return f'color: {ALERT_RED}; font-weight: bold' if v < 0.5 else ''

_styled = (
    summary_df.style
    .format(_fmt_cell)
    .map(_highlight_low)
    .set_caption('BioKGSuite - full benchmark summary (values < 0.5 in red)')
    .set_table_styles([
        {'selector': 'caption',
         'props': [('font-size', '11pt'), ('font-weight', 'bold'),
                   ('color', TEXT_COLOR), ('caption-side', 'top')]},
        {'selector': 'th',
         'props': [('font-size', '8pt'), ('text-align', 'center'),
                   ('padding', '4px 6px'), ('white-space', 'nowrap')]},
        {'selector': 'td',
         'props': [('font-size', '8.5pt'), ('text-align', 'center'),
                   ('padding', '3px 6px')]},
    ])
)
_styled

## 6. Concordance among metrics

*Does a general quality ordering exist across the graphs in the panel?*

Each metric is treated as a judge ranking the six KGs, and agreement among judges is
summarised by **Kendall's coefficient of concordance** $W$, which is 0 when judges
disagree completely and 1 under perfect agreement. $W$ converts exactly into the mean
pairwise Spearman correlation between judges, $\bar\rho = (kW - 1)/(k - 1)$ for $k$
judges, which is the interpretable quantity reported in the text.

**Null model.** KG labels are permuted independently within each metric, which destroys
any consistent ordering of graphs while preserving each metric's rank structure. Note
that $E[W] = 1/k$ under this null, so $W$ has a positive floor that rises as judges are
removed. Because the metrics are not independent judges (the three task-performance
metrics share a scoring heuristic, and the coverage metrics share gold standards), the
permutation *p*-value is optimistic and the **effect size and its stability across
specifications are the quantities to report**, not the *p*-value.

**Specifications.** $W$ is recomputed under five specifications to establish that the
estimate is not an artefact of metric selection: all metrics, only those that
discriminate between any two graphs, the seven aggregated dimension scores, every
possible choice of one metric per dimension (enumerated exhaustively), and
leave-one-metric-out jackknife.

**Outputs:** `results/tables/00_summary/concordance_specifications.csv` ·
`results/tables/00_summary/rank_dispersion.csv` ·
`results/tables/00_summary/dimension_dissociations.csv`

In [ ]:
# Kendall's W with a within-metric permutation null.
_RNG_W   = np.random.default_rng(0)
_N_PERM  = 20_000
# A metric "discriminates" if its range across the graphs exceeds this. The
# value is a floor on what counts as a real difference rather than rounding,
# chosen a priori at 5% of the 0-1 scale every sub-metric is reported on; the
# jackknife and one-per-dimension specifications below show what the estimate
# does when the set it selects is perturbed.
_DISC_TOL = 0.05

# metric -> dimension map, taken from SUB_METRICS so it cannot drift from the table
_METRIC_DIM = {_lab: _dim for _lab, _dim, _ in SUB_METRICS}
_METRIC_COLS = [_lab for _lab, _, _ in SUB_METRICS]


def _rank_matrix(df, cols):
    """KGs x metrics matrix of ranks (1 = best). Average ranks for ties."""
    return df[cols].rank(ascending=False, method='average')


def kendall_w(df, cols):
    """Kendall's W over the KGs, treating each column as a judge."""
    _R  = _rank_matrix(df, cols).dropna(axis=1)      # drop judges with missing scores
    _Rt = _R.T.to_numpy()                            # judges x objects
    _k, _n = _Rt.shape
    if _k < 2 or _n < 2:
        return np.nan, np.nan, np.nan, _k
    _s = _Rt.sum(axis=0)
    _W = 12 * ((_s - _s.mean()) ** 2).sum() / (_k ** 2 * (_n ** 3 - _n))
    _rho = (_k * _W - 1) / (_k - 1)                  # mean pairwise Spearman
    return _W, _rho, _Rt, _k


def perm_p(_Rt, _W, n_perm=_N_PERM, rng=_RNG_W):
    """P(W_null >= W_obs) under permutation of KG labels within each metric."""
    _k, _n = _Rt.shape
    _null = np.empty(n_perm)
    for _i in range(n_perm):
        _P = np.apply_along_axis(rng.permutation, 1, _Rt)
        _s = _P.sum(axis=0)
        _null[_i] = 12 * ((_s - _s.mean()) ** 2).sum() / (_k ** 2 * (_n ** 3 - _n))
    return float((_null >= _W).mean()), float(_null.mean())


# ---------------------------------------------------------------- specifications
# Concordance is a statement about how the metrics order the graphs, so a graph
# no dimension scored is absent rather than tied. Dropping it here keeps it from
# arriving as a row of NaN, which kendall_w would answer by discarding every
# judge that has one.
scored_df  = summary_df.dropna(how='all')
_unscored  = [_kg for _kg in summary_df.index if _kg not in scored_df.index]
if len(scored_df) < 2:
    raise RuntimeError(
        f'Concordance needs at least two scored graphs; {len(scored_df)} present.')
if _unscored:
    print('Unscored graphs, excluded here: ' + ', '.join(_unscored))

_avail = [_c for _c in _METRIC_COLS if _c in scored_df.columns]
_disc  = [_c for _c in _avail
          if scored_df[_c].notna().sum() >= 2
          and (scored_df[_c].max() - scored_df[_c].min()) > _DISC_TOL]
_dimcols = [_c for _c in scored_df.columns if _c.endswith(' [dim]')]

_specs = []
for _label, _cols in [('All metrics', _avail),
                      ('Discriminating metrics', _disc),
                      ('Dimension scores', _dimcols)]:
    _W, _rho, _Rt, _k = kendall_w(scored_df, _cols)
    _p, _nullW = perm_p(_Rt, _W)
    _specs.append(dict(specification=_label, n_judges=_k, W=round(_W, 3),
                       mean_rho=round(_rho, 3), null_W=round(_nullW, 3),
                       perm_p=_p, note=''))

# One metric per dimension, every combination enumerated
_by_dim = {}
for _c in _disc:
    _by_dim.setdefault(_METRIC_DIM[_c], []).append(_c)
_combos = list(product(*_by_dim.values()))
_Wc = np.array([kendall_w(scored_df, list(_cb))[0] for _cb in _combos])
_specs.append(dict(specification='One metric per dimension', n_judges=len(_by_dim),
                   W=round(float(np.median(_Wc)), 3),
                   mean_rho=round(float((len(_by_dim) * np.median(_Wc) - 1) / (len(_by_dim) - 1)), 3),
                   null_W=round(1 / len(_by_dim), 3), perm_p=np.nan,
                   note=f'median of {len(_combos)} combinations, '
                        f'range {_Wc.min():.3f}-{_Wc.max():.3f}'))

# Leave-one-metric-out jackknife
_Wj = np.array([kendall_w(scored_df, [_x for _x in _disc if _x != _m])[0] for _m in _disc])
_specs.append(dict(specification='Leave-one-metric-out', n_judges=len(_disc) - 1,
                   W=round(float(np.median(_Wj)), 3),
                   mean_rho=round(float(((len(_disc) - 1) * np.median(_Wj) - 1) / (len(_disc) - 2)), 3),
                   null_W=round(1 / (len(_disc) - 1), 3), perm_p=np.nan,
                   note=f'median over {len(_disc)} refits, '
                        f'range {_Wj.min():.3f}-{_Wj.max():.3f}'))

concordance_df = pd.DataFrame(_specs)
out.table(concordance_df, 'concordance_specifications')
_all_W = np.concatenate([[_s['W'] for _s in _specs], _Wc, _Wj])
print(concordance_df.to_string(index=False))
print(f"\nW across all specifications: {_all_W.min():.2f} to {_all_W.max():.2f}")
print(f"Headline (discriminating metrics): W = {_specs[1]['W']:.2f}, "
      f"mean pairwise rho = {_specs[1]['mean_rho']:.2f}, "
      f"permutation p = {_specs[1]['perm_p']:.4f}")

### Rank dispersion

How far each graph's rank moves across the discriminating metrics. A single quality ordering would hold every graph near one rank.

In [ ]:
# Rank dispersion within each KG across the discriminating metrics
_R = _rank_matrix(scored_df, _disc)
_disp_rows = []
for _kg in _R.index:
    _v = _R.loc[_kg].dropna()
    _disp_rows.append(dict(KG=_kg,
                      best_rank=int(_v.min()), worst_rank=int(_v.max()),
                      rank_span=int(_v.max() - _v.min()),
                      n_first=int((_v == 1).sum()),
                      n_top2=int((_v <= 2).sum()),
                      n_bottom2=int((_v >= len(_R) - 1).sum()),
                      n_metrics=int(_v.size)))
dispersion_df = pd.DataFrame(_disp_rows).sort_values('rank_span', ascending=False)
print(dispersion_df.to_string(index=False))
print(f"\nEvery KG spans at least {dispersion_df.rank_span.min()} of "
      f"{len(_R)} available ranks.")
_lead = dispersion_df.set_index('KG').n_first
print("Metrics led, by KG: " + ", ".join(f"{_k} {_v}" for _k, _v in _lead.items()))

out.table(dispersion_df, 'rank_dispersion')

### Dissociation count

The fraction of (graph pair x property pair) comparisons whose sign reverses. It assumes nothing about the distribution of the scores, which is what makes it readable at this panel size.

In [ ]:
# Dissociation count: the assumption-free test of non-redundancy. It asks only
# whether two properties order a pair of graphs the same way, so it needs no
# distributional assumption and no null, which matters at this panel size.

def dissociation_rate(cols):
    """Fraction of (graph pair x property pair) comparisons whose sign reverses.

    0% means the properties order the graphs identically (fully redundant);
    50% is the expectation if the two orderings are unrelated. Ties are excluded.
    """
    _tot = _rev = 0
    for _a, _b in combinations(scored_df.index, 2):
        for _p, _q in combinations(cols, 2):
            _dp = scored_df.loc[_a, _p] - scored_df.loc[_b, _p]
            _dq = scored_df.loc[_a, _q] - scored_df.loc[_b, _q]
            if pd.isna(_dp) or pd.isna(_dq) or _dp == 0 or _dq == 0:
                continue
            _tot += 1
            _rev += (_dp > 0) != (_dq > 0)
    return _rev, _tot, _rev / _tot if _tot else np.nan

for _label, _cols in [('Dimension scores', _dimcols), ('Discriminating metrics', _disc)]:
    _r, _t, _f = dissociation_rate(_cols)
    print(f'{_label:24s} {_r:5d} reversals of {_t:5d} comparisons = {_f:.1%}')

# Per dimension pair, to identify any pair that does track together
_diss_rows = []
for _p, _q in combinations(_dimcols, 2):
    _r = _t = 0
    for _a, _b in combinations(scored_df.index, 2):
        _dp = scored_df.loc[_a, _p] - scored_df.loc[_b, _p]
        _dq = scored_df.loc[_a, _q] - scored_df.loc[_b, _q]
        if pd.isna(_dp) or pd.isna(_dq) or _dp == 0 or _dq == 0:
            continue
        _t += 1
        _rev = (_dp > 0) != (_dq > 0)
        _r += _rev
    _diss_rows.append(dict(dimension_1=_p.replace(' [dim]', ''),
                      dimension_2=_q.replace(' [dim]', ''),
                      reversals=_r, comparisons=_t,
                      fraction=round(_r / _t, 3) if _t else np.nan))
dissociation_df = pd.DataFrame(_diss_rows).sort_values('fraction', ascending=False)
print()
print(dissociation_df.to_string(index=False))
print(f"\n{(dissociation_df.fraction >= 1/3).sum()} of {len(dissociation_df)} dimension "
      "pairs reverse on at least a third of graph pairs.")
out.table(dissociation_df, 'dimension_dissociations')
print(out.summary())